In [ ]:
#Week 4-3
#Bolortulga Seded
#8th of March
#24045487
#Feature Encoding - adult dataset

In [11]:
import pandas as pd
import numpy as np
import sklearn.preprocessing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

df = pd.read_csv('adult.csv')

target_name = "income"
target = df[target_name]
data = df.drop(columns=[target_name])

categorical_columns = categorical_columns = ['workclass', 'education', 'marital.status', 'occupation', 
                                             'relationship', 'race', 'sex', 'native.country']
data_categorical = data[categorical_columns]
print(data_categorical.shape)
print(data_categorical.head())

(32561, 8)
  workclass     education marital.status         occupation   relationship  \
0         ?       HS-grad        Widowed                  ?  Not-in-family   
1   Private       HS-grad        Widowed    Exec-managerial  Not-in-family   
2         ?  Some-college        Widowed                  ?      Unmarried   
3   Private       7th-8th       Divorced  Machine-op-inspct      Unmarried   
4   Private  Some-college      Separated     Prof-specialty      Own-child   

    race     sex native.country  
0  White  Female  United-States  
1  White  Female  United-States  
2  Black  Female  United-States  
3  White  Female  United-States  
4  White  Female  United-States  


In [45]:
print(data.dtypes)
print("\nnative.country value counts:")
print(data['native.country'].value_counts().sort_index())

age               int64
workclass           str
fnlwgt            int64
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
sex                 str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
dtype: object

native.country value counts:
native.country
?                               583
Cambodia                         19
Canada                          121
China                            75
Columbia                         59
Cuba                             95
Dominican-Republic               70
Ecuador                          28
El-Salvador                     106
England                          90
France                           29
Germany                         137
Greece                           29
Guatemala                        64
Haiti                            44
Holand-Netherlands                1
Honduras                     

In [37]:
print(data.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country']


In [47]:
le = LabelEncoder()
df_label = df.copy()
df_label['income_encoded'] = le.fit_transform(df_label['income'])

print("Class lables mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df_label[['income', 'income_encoded']].head())

Class lables mapping: {'<=50K': np.int64(0), '>50K': np.int64(1)}
  income  income_encoded
0  <=50K               0
1  <=50K               0
2  <=50K               0
3  <=50K               0
4  <=50K               0


In [48]:
education_column = data_categorical[['education']]

encoder = OrdinalEncoder().set_output(transform="pandas")
education_encoded = encoder.fit_transform(education_column) 
print(education_encoded.head())
print("Categories:", encoder.categories_)

   education
0       11.0
1       11.0
2       15.0
3        5.0
4       15.0
Categories: [array(['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th', '9th',
       'Assoc-acdm', 'Assoc-voc', 'Bachelors', 'Doctorate', 'HS-grad',
       'Masters', 'Preschool', 'Prof-school', 'Some-college'],
      dtype=object)]


In [49]:
data_encoded = encoder.fit_transform(data_categorical)
data_encoded[:5]
print(f"The Dataset encoded contains {data_encoded.shape[1]} features")

The Dataset encoded contains 8 features


In [51]:
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
education_encoded_ohe = encoder.fit_transform(education_column)
print(education_encoded_ohe.head())

   education_10th  education_11th  education_12th  education_1st-4th  \
0             0.0             0.0             0.0                0.0   
1             0.0             0.0             0.0                0.0   
2             0.0             0.0             0.0                0.0   
3             0.0             0.0             0.0                0.0   
4             0.0             0.0             0.0                0.0   

   education_5th-6th  education_7th-8th  education_9th  education_Assoc-acdm  \
0                0.0                0.0            0.0                   0.0   
1                0.0                0.0            0.0                   0.0   
2                0.0                0.0            0.0                   0.0   
3                0.0                1.0            0.0                   0.0   
4                0.0                0.0            0.0                   0.0   

   education_Assoc-voc  education_Bachelors  education_Doctorate  \
0                 

In [57]:
encoder_ohe_full = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
data_encoded_ohe = encoder_ohe_full.fit_transform(data_categorical)

print(data_encoded_ohe[:5])
print(f"The encoded dataset contains {data_encoded_ohe.shape[1]} features")

   workclass_?  workclass_Federal-gov  workclass_Local-gov  \
0          1.0                    0.0                  0.0   
1          0.0                    0.0                  0.0   
2          1.0                    0.0                  0.0   
3          0.0                    0.0                  0.0   
4          0.0                    0.0                  0.0   

   workclass_Never-worked  workclass_Private  workclass_Self-emp-inc  \
0                     0.0                0.0                     0.0   
1                     0.0                1.0                     0.0   
2                     0.0                0.0                     0.0   
3                     0.0                1.0                     0.0   
4                     0.0                1.0                     0.0   

   workclass_Self-emp-not-inc  workclass_State-gov  workclass_Without-pay  \
0                         0.0                  0.0                    0.0   
1                         0.0           

In [59]:
education_order = [['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th',
                    '10th', '11th', '12th', 'HS-grad', 'Some-college',
                    'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
                    'Prof-school', 'Doctorate']]
oe = OrdinalEncoder(categories=education_order
                    , handle_unknown='use_encoded_value', unknown_value=-1)
df_ord = df.copy()
df_ord['education_ord'] = oe.fit_transform(df_ord[['education']])
print(df_ord[['education', 'education_ord']].drop_duplicates()
      .sort_values('education_ord'))


         education  education_ord
1106     Preschool            0.0
26         1st-4th            1.0
27         5th-6th            2.0
3          7th-8th            3.0
197            9th            4.0
6             10th            5.0
16            11th            6.0
178           12th            7.0
0          HS-grad            8.0
2     Some-college            9.0
25       Assoc-voc           10.0
18      Assoc-acdm           11.0
12       Bachelors           12.0
13         Masters           13.0
11     Prof-school           14.0
7        Doctorate           15.0


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

ordinal_features = ['education']
ordinal_categories = [['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th',
                    '10th', '11th', '12th', 'HS-grad', 'Some-college',
                    'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
                    'Prof-school', 'Doctorate']]
nominal_features = ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=ordinal_categories, 
                                        handle_unknown='use_encoded_value',
                                        unknown_value=-1), ordinal_features),
                                           
                ('nom', OneHotEncoder(sparse_output=False,
                                      handle_unknown='ignore'),nominal_features)]
)
X = data_categorical
X_prepared = preprocessor.fit_transform(X)
print("Transformed shape:", X_prepared.shape)


Transformed shape: (32561, 87)


In [13]:
print(data_categorical.shape)

(32561, 8)


In [19]:
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(target)

final_df = pd.DataFrame(
    np.hstack([X_prepared, y_encoded.reshape(-1,1)]),
    columns=
    list(preprocessor.get_feature_names_out()) + ['income_encoded']
)
print(final_df.head())
print("Final shape:", final_df.shape)
    


   ord__education  nom__workclass_?  nom__workclass_Federal-gov  \
0             8.0               1.0                         0.0   
1             8.0               0.0                         0.0   
2             9.0               1.0                         0.0   
3             3.0               0.0                         0.0   
4             9.0               0.0                         0.0   

   nom__workclass_Local-gov  nom__workclass_Never-worked  \
0                       0.0                          0.0   
1                       0.0                          0.0   
2                       0.0                          0.0   
3                       0.0                          0.0   
4                       0.0                          0.0   

   nom__workclass_Private  nom__workclass_Self-emp-inc  \
0                     0.0                          0.0   
1                     1.0                          0.0   
2                     0.0                          0.0   
3   

In [20]:
model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=500)
)
cv_results = cross_validate(model, data_categorical, target)
scores = cv_results['test_score']
print(f"The accuracy is: {scores.mean():.3f} ± {scores.std():.3f}")

The accuracy is: 0.830 ± 0.006
